# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

# 📥 Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Test Accuracy:", ann_test_acc)

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print("CNN Test Accuracy:", cnn_test_acc)

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(ann_history.history['val_accuracy'], label='ANN Val Acc')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN Validation Accuracy")
plt.legend()
plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# Suggested optional run:
# aug_history = aug_cnn_model.fit(x_train_norm, y_train, epochs=10, validation_split=0.1)

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc]
})
comparison

# 🎓 Student Learning Tasks
Try these tasks after understanding the notebook:

### ✅ Beginner Tasks
1. Increase ANN layers and observe performance
2. Change CNN filters from 32→64→128
3. Increase epochs to 20
4. Add **EarlyStopping**
5. Add **data augmentation training**

## ✅ Task 1: Increase ANN Layers and Observe Performance

We add **more Dense layers** (1024 → 512 → 256) with Dropout after each block. A deeper ANN has
more capacity, but since it still sees the image as a flat 3072-length vector, we expect only a
**small improvement** over the baseline ANN — depth cannot compensate for the loss of spatial
structure. This is the key observation of this task.

In [ ]:
# Task 1: Deeper ANN — more Dense layers than the baseline (512 -> 256)
deep_ann_model = models.Sequential([
    layers.Dense(1024, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

deep_ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

deep_ann_history = deep_ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

deep_ann_test_loss, deep_ann_test_acc = deep_ann_model.evaluate(x_test_flat, y_test)
print("Deeper ANN Test Accuracy:", deep_ann_test_acc)
print("Baseline ANN Test Accuracy:", ann_test_acc)

**Observation (Task 1):** Even with roughly 3× more parameters, the deeper ANN improves only
marginally (and can even overfit faster). Flattening the image destroys the neighborhood
relationships between pixels, so extra Dense layers mostly memorize instead of learning
better visual features. This motivates convolution.

## ✅ Tasks 2, 3, 4 & 5: Scaled-Up Augmented CNN with EarlyStopping

We combine the remaining four tasks into one **final improved model**, since together they form a
realistic training-strategy upgrade:

- **Task 2 — Scale up filters 32 → 64 → 128:** three convolutional blocks whose filter counts
  double at each stage, so deeper layers can encode progressively more complex patterns
  (edges → textures → object parts).
- **Task 3 — Train for 20 epochs:** doubling the epoch budget gives the augmented model time to
  converge (augmented data is harder to fit, so it learns more slowly but generalizes better).
- **Task 4 — EarlyStopping:** monitors `val_accuracy` with `patience=4` and
  `restore_best_weights=True`, so if validation performance stops improving, training halts and
  the best weights are kept — an automatic guard against overfitting and wasted compute.
- **Task 5 — Data augmentation training run:** the `RandomFlip` / `RandomRotation` / `RandomZoom`
  pipeline defined earlier is now **actually trained**, generating new transformed images every
  epoch so the network never sees the exact same picture twice.

In [ ]:
# Tasks 2-5: augmented CNN, filters scaled 32 -> 64 -> 128,
# 20 epochs, EarlyStopping, augmentation pipeline actively trained
early_stop = tf.keras.callbacks.EarlyStopping(          # Task 4
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

final_cnn_model = models.Sequential([
    data_augmentation,                                   # Task 5 (RandomFlip/Rotation/Zoom)

    layers.Conv2D(32, (3,3), activation='relu', padding='same',
                  input_shape=(32,32,3)),                # Task 2: 32 filters
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),   # Task 2: 64 filters
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),  # Task 2: 128 filters
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

final_cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

final_history = final_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,                                           # Task 3
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop]
)

final_test_loss, final_test_acc = final_cnn_model.evaluate(x_test_norm, y_test)
print("Augmented CNN (Tasks 2-5) Test Accuracy:", final_test_acc)

## 📈 Validation Curves — All Model Variants

One chart with every variant makes the architecture story obvious: both ANNs plateau early at a
much lower accuracy, the baseline CNN jumps well above them, and the augmented CNN climbs more
slowly at first (augmented images are harder) but ends highest and with the smallest
train/validation gap.

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(ann_history.history['val_accuracy'], label='ANN (baseline)')
plt.plot(deep_ann_history.history['val_accuracy'], label='Deeper ANN (Task 1)')
plt.plot(cnn_history.history['val_accuracy'], label='CNN (baseline)')
plt.plot(final_history.history['val_accuracy'], label='Augmented CNN (Tasks 2-5)')
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy — All Model Variants")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 📊 Extended Final Comparison Table

Test accuracy for **all four model variants**, sorted from weakest to strongest.

In [ ]:
extended_comparison = pd.DataFrame({
    "Model": ["ANN (baseline)", "Deeper ANN (Task 1)",
              "CNN (baseline)", "Augmented CNN (Tasks 2-5)"],
    "Test Accuracy": [ann_test_acc, deep_ann_test_acc,
                      cnn_test_acc, final_test_acc]
}).sort_values("Test Accuracy").reset_index(drop=True)

extended_comparison

## 📝 Observations from the Experiments

1. **Architecture matters more than depth.** The deeper ANN (Task 1) barely improves on the
   baseline ANN despite far more parameters — flat vectors simply discard the spatial structure
   that makes an image an image.
2. **Convolution is the real jump.** The baseline CNN, with *fewer* parameters than the deep ANN,
   outperforms both ANNs by a wide margin because Conv2D layers learn translation-aware local
   features and reuse them across the whole image.
3. **Scaling filters 32→64→128 builds a feature hierarchy.** Early layers capture edges and
   colors; later, wider layers combine them into textures and object parts.
4. **Augmentation trades speed for generalization.** The augmented CNN's training accuracy rises
   more slowly (every epoch shows different flips/rotations/zooms), but its validation and test
   accuracy end the highest, with the smallest overfitting gap.
5. **EarlyStopping makes the 20-epoch budget safe.** If validation accuracy stalls, training stops
   and the best weights are restored automatically — we get the benefit of a longer run without
   the risk of over-training.

# ✅ Conclusion
- **ANN works**, but ignores image structure
- **CNN extracts spatial features**, so it performs significantly better
- **Training strategies** like dropout, batch norm, and augmentation improve results
- This project builds strong fundamentals for **computer vision interviews and deep learning projects**